# 🤖 JINXMP3 AI-OS Brain
**This notebook runs the AI model that controls your entire laptop remotely.**

Steps:
1. Run Cell 1 — install packages
2. Run Cell 2 — paste your ngrok token
3. Run Cell 3 — start the server
4. Copy the public URL into your laptop `.env` as `COLAB_URL`
5. Run `python main.py --backend colab` on your laptop

Your laptop is now controlled by Colab's free GPU 🚀

In [ ]:
# Cell 1 — Install (run once)
!pip install flask pyngrok transformers accelerate bitsandbytes sentencepiece -q
!nvidia-smi  # Check GPU

In [ ]:
# Cell 2 — Config
# Get free ngrok token at: https://ngrok.com/signup
NGROK_TOKEN = ''  # <-- paste your token here

# Choose model (bigger = smarter but slower to load)
# Free T4 GPU options:
MODEL_ID = 'microsoft/Phi-3.5-mini-instruct'  # Best quality for free tier (~7GB)
# MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'  # Ultra fast, tiny
# MODEL_ID = 'google/gemma-2-2b-it'               # Google's model
# MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3' # Needs Colab Pro

print(f'Model: {MODEL_ID}')
print('Ready for Cell 3')

In [ ]:
# Cell 3 — Load model and start server
import torch
from flask import Flask, request, jsonify
from threading import Thread
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from pyngrok import ngrok
import json

print('Loading model... (this takes 1-3 minutes first time)')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    load_in_4bit=True  # 4-bit quantization = fits in free T4
)
pipe = pipeline(
    'text-generation', model=model, tokenizer=tokenizer,
    max_new_tokens=1024, temperature=0.3, do_sample=True,
    return_full_text=False
)
print('Model loaded!')

# ── Flask API server ──────────────────────────────────────────
app = Flask('ai-os-brain')

def messages_to_prompt(messages):
    """Convert OpenAI-style messages to a text prompt."""
    prompt = ''
    for m in messages:
        role = m.get('role', 'user')
        content = m.get('content', '')
        if role == 'system':
            prompt += f'<|system|>\n{content}<|end|>\n'
        elif role == 'user':
            prompt += f'<|user|>\n{content}<|end|>\n'
        elif role == 'assistant':
            prompt += f'<|assistant|>\n{content}<|end|>\n'
    prompt += '<|assistant|>\n'
    return prompt

@app.get('/health')
def health():
    return jsonify({'status': 'ok', 'model': MODEL_ID, 'gpu': torch.cuda.is_available()})

@app.post('/chat')
def chat():
    try:
        data = request.json
        messages = data.get('messages', [])
        prompt = messages_to_prompt(messages)
        result = pipe(prompt)
        response = result[0]['generated_text'].strip()
        return jsonify({'response': response})
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.post('/api/chat')  # OpenAI-compatible endpoint
def openai_chat():
    try:
        data = request.json
        messages = data.get('messages', [])
        prompt = messages_to_prompt(messages)
        result = pipe(prompt, max_new_tokens=data.get('max_tokens', 1024))
        response = result[0]['generated_text'].strip()
        return jsonify({
            'choices': [{'message': {'role': 'assistant', 'content': response}}],
            'model': MODEL_ID
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 500

# Start server in background thread
Thread(target=lambda: app.run(port=5000, use_reloader=False), daemon=True).start()
print('Server running on port 5000')

# ── Expose via ngrok ──────────────────────────────────────────
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

tunnel = ngrok.connect(5000)
public_url = tunnel.public_url

print()
print('=' * 60)
print('  JINXMP3 AI-OS BRAIN IS LIVE')
print(f'  URL: {public_url}')
print('=' * 60)
print()
print('On your laptop, run:')
print(f'  export COLAB_URL={public_url}')
print(f'  cd ~/Ai-operating-system')
print(f'  source .venv/bin/activate')
print(f'  python main.py --backend colab --both')
print()
print('Your entire computer is now controlled by Colab GPU!')

In [ ]:
# Cell 4 (optional) — Keep alive
# Colab disconnects after ~90 min of inactivity.
# This cell sends a heartbeat every 60 seconds.
import time
import requests

print('Keeping session alive... (Ctrl+C to stop)')
while True:
    try:
        r = requests.get('http://localhost:5000/health', timeout=5)
        print(f'Heartbeat: {r.json()["status"]} | GPU: {r.json()["gpu"]}')
    except Exception as e:
        print(f'Heartbeat failed: {e}')
    time.sleep(60)